# 06 · clustering and label transfer

You arrive with a Harmonized lipid embedding and two coronal sections of the same brain plane, one
control and one pregnant, and not a single group label yet. This notebook turns that embedding into
**lipid territories** and gives every pixel a name. The path has three movements, then a fourth that
shows you the production engine.

First we **cluster**. We build a graph of who-neighbours-whom in the lipid embedding and let a
community-finder cut the pixels into groups. Those groups are the **lipizones**, the data-driven
territories of lipid composition the Lipid Brain Atlas is built on. You will get *your own* lipizones,
learned from your own embedding, and they will not be identical to the paper's. That is the honest
situation of a real analysis, and the broad biology survives it.

Second we **compare to anatomy**. We colour the section by lipizone, lay it next to the Allen reference,
and measure with one number, the Adjusted Rand Index, how much the lipid territories agree with
classical anatomy. The answer is the heart of the lesson: they overlap, but lipizones are their own
thing.

Third we **transfer labels**. We learn lipizones on the control brain, then carry them onto the
pregnant brain by asking each pregnant pixel which control pixels it sits next to in the shared space.
It is the same learn-then-transfer logic the Atlas uses to annotate new brains from few sections,
except the Atlas does the transfer with EUCLID's trained classifiers (the fourth movement below), and it
is what lets us put control and pregnant side by side speaking one vocabulary.

We close with the real thing. You git-clone **EUCLID**, the package behind the Atlas, and actually run
its clustering on your control brain, so you can put EUCLID's lipizones next to the ones you built by
hand and see that the package does a more careful, divisive version of the same idea.

One running rule for today: every time you call a `cl.…` helper, **open** `src/cajal_lipidomics/<module>.py` and **read** that function first. The whole course is built so you can see the code behind every number; never run a helper you have not read.

In [ ]:
# the stack we use today, plus the course helpers and the lab figure style
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import KNeighborsClassifier

import cajal_lipidomics as cl
from cajal_lipidomics import embedding as emb, analysis as an, plotting as pl
from cajal_lipidomics.style import set_style, FS
set_style()

# one seed for everything random today, so the notebook is reproducible
RNG_SEED = 0
np.random.seed(RNG_SEED)

print("ready: numpy", np.__version__, "| scanpy", sc.__version__)

**check:** you see `ready: numpy ... | scanpy ...` and no red error. `set_style()` applies the lab
defaults so every figure comes out clean and Illustrator-editable without you fussing over it.

## 1. load the embedded sections and pull out the factors

We pick up where the embedding notebook left off, loading `05_embedded.h5ad`. It is one AnnData holding
both coronal sections of the same plane (anterior-posterior level around 6.5): one control female
(`Condition == "naive"`, `SectionID 75`) and one pregnant at embryonic day 13.5 (`Condition ==
"pregnant"`, `SectionID 110`). Each pixel is one MALDI desorption spot, a tiny laser point of tissue,
not one cell. It mixes cell bodies, axons, dendrites, and glia.

`adata.layers["umaia"]` holds the uMAIA-normalized lipid intensities. `adata.obs` carries the per-pixel
metadata: section and condition, the Allen CCF coordinates `zccf`/`yccf`, and the Allen anatomical
`acronym` of the region each pixel falls in, with its official colour `allencolor`.

The ingredient we actually cluster on lives in `adata.obsm`. The embedding notebook compressed the
lipids into **12 NMF factors** (`X_nmf`), additive, non-negative **lipid programs**, then ran **Harmony**
to remove the section-to-section offset, giving `X_harmony`. We cluster on `X_harmony`, the 12 summary
coordinates per pixel, never on the raw lipids directly. The paper fits 16 such factors on the whole-brain
atlas; on our two sections the embedding notebook settled on 12. The exact count is a choice, not a law:
it is the number of lipid programs rich enough to describe this data without fitting its noise.

In [ ]:
# load the embedded substrate produced by the previous notebook
adata = ad.read_h5ad("../../data/derived/05_embedded.h5ad")
print("pixels x lipids:", adata.shape)

# the two embeddings: raw NMF programs, and the Harmony-corrected version we cluster on
G_nmf = adata.obsm["X_nmf"].astype(float)        # parts-based lipid programs, per pixel
G = adata.obsm["X_harmony"].astype(float)        # same, with the section offset removed
print("NMF embedding:", G_nmf.shape, "| Harmonized embedding:", G.shape)

# boolean masks for the two halves of the data
is_control  = (adata.obs["Condition"] == "naive").to_numpy()
is_pregnant = (adata.obs["Condition"] == "pregnant").to_numpy()
print(f"control pixels: {is_control.sum():>6}   pregnant pixels: {is_pregnant.sum():>6}")

**check:** `pixels x lipids: (189011, 104)`, both embeddings `(189011, 12)`, and roughly 88753
control plus 100258 pregnant pixels. The 12-factor `G` is what we cluster and transfer on; the masks
split it into the reference (control) and the query (pregnant).

Before clustering, look at the embedding. Each factor is a lipid program, and its value at a
pixel says how strongly that program is expressed there. Painting a few factors back onto the brain
coordinates is the fastest way to see these numbers are not random: they carry sharp anatomical
structure, which is exactly what makes them clusterable.

In [ ]:
# paint the first four NMF factors onto the control section's CCF coordinates
z = adata.obs["zccf"].to_numpy(); y = -adata.obs["yccf"].to_numpy()   # flip y so dorsal is up
m = is_control                                                        # one section for clarity

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for k, ax in enumerate(axes):
    v = G_nmf[:, k]
    ax.scatter(z[m], y[m], c=v[m], cmap="plasma", s=3, rasterized=True,
               vmin=np.quantile(v[m], 0.02), vmax=np.quantile(v[m], 0.98))
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"NMF factor {k}", fontsize=FS["m"])
    for s in ax.spines.values(): s.set_visible(False)
fig.suptitle("four lipid programs painted on the control section", fontsize=FS["l"])
plt.tight_layout(); plt.show()

**check:** each panel shows crisp territories, not noise. One factor lights up the fibre tracts,
another the cortical ribbon, another a deep nucleus. The factors already *are* a coarse map of the
brain, and clustering will sharpen these continuous programs into discrete, named regions.

❓ **QUESTION.** Why cluster on the 12 factors instead of the 104 raw lipids? Two reasons hide in the
plot. The factors are already de-noised and de-batched (Harmony ran on them), and 12 numbers make the
neighbour graph far cheaper to build than 104. Clustering on the embedding is clustering on the
*signal*, with the acquisition artifact and the redundancy already removed.

## 2. the kNN graph: write down who-is-near-whom

We want to carve the pixels into groups without telling the algorithm how many to expect. The first
step is to turn the cloud of 12-dimensional points into a **graph**, a network of connections.

The idea, in one sentence: for each pixel find its **k nearest neighbours** in the embedding, the k
other pixels whose factors are most similar, and draw an edge to each. Do this for every pixel and you
get a network where pixels with similar lipid composition are wired together and pixels with different
composition are not. "Distance" here is ordinary Euclidean distance in the 12-dimensional factor space,
the straight-line distance you would compute with Pythagoras, just in twelve dimensions.

That graph is the entire input to community detection. It throws away the exact coordinates and keeps
only the relationships. Two pixels on opposite sides of the brain that happen to share a lipid mix end
up neighbours in the graph even though they are far apart physically, which is the point: we group by
lipid identity, not by location.

Let us build it on a small random sample first, so you see the graph is just a table of edges, before
we hand the full job to scanpy.

In [ ]:
# build a kNN graph BY HAND on a small sample, to see there is no magic in it
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(RNG_SEED)
sample = rng.choice(np.where(is_control)[0], size=2000, replace=False)   # 2000 control pixels
Gs = G[sample]

k = 15                                                  # 15 nearest neighbours per pixel
nn = NearestNeighbors(n_neighbors=k + 1).fit(Gs)        # +1 because the closest point is itself
dist, idx = nn.kneighbors(Gs)
dist, idx = dist[:, 1:], idx[:, 1:]                     # drop the self-match in column 0

print("each of", Gs.shape[0], "pixels is linked to its", k, "nearest neighbours")
print("so the graph has", Gs.shape[0] * k, "directed edges")
print("example: pixel 0's neighbours are rows", idx[0][:5], "...")
print("their distances in factor space:", np.round(dist[0][:5], 4))

**check:** you built a graph, 2000 pixels each linked to its 15 nearest, so 30000 edges. `idx[0]`
lists the row numbers of pixel 0's neighbours and `dist[0]` their distances. That edge list *is* the
graph, nothing more mysterious than a table of who-is-near-whom.

A picture makes it concrete. We lay the 2000 sampled pixels out by two of their factors and draw
a line for every edge. Dense thickets of lines are pixels that all resemble each other, the future
clusters; sparse regions are the gaps between groups.

In [ ]:
# draw the kNN graph: points placed by two factors, a faint line per edge
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for i in range(Gs.shape[0]):
    for j in idx[i]:
        ax.plot([Gs[i, 0], Gs[j, 0]], [Gs[i, 1], Gs[j, 1]],
                lw=0.1, color="0.6", alpha=0.3, rasterized=True)
ax.scatter(Gs[:, 0], Gs[:, 1], s=6, c="crimson", rasterized=True, zorder=3)
ax.set_xlabel("Harmonized factor 0"); ax.set_ylabel("Harmonized factor 1")
ax.set_title("kNN graph on 2000 control pixels (k=15)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

**check:** clumps of densely connected red points joined by a haze of thin grey lines, with
thinner bridges between clumps. The community-finder we run next is just an algorithm that spots those
clumps automatically.

💡 **HINT.** We plotted only factors 0 and 1, so two pixels can *look* far apart on screen yet be close
neighbours through the other ten dimensions. The graph is honest about all twelve; the scatter is a
flattened shadow of it. That is also why a 2D plot can never fully justify a clustering: trust the
graph, use the picture as intuition.

## 3. Leiden: cut the graph into communities

Now the graph gets cut into groups. **Leiden** is a community-detection algorithm: it scans the graph
and finds sets of pixels densely wired to each other and only sparsely wired to the rest, exactly how
you would spot friend-groups in a social network. Each community becomes one cluster. It never sees
coordinates or anatomy, only the edges, and it decides the number of clusters on its own.

The score Leiden maximizes is **modularity**: roughly, the fraction of edges that fall *inside*
communities minus what you would expect if the same edges were thrown down at random. A grouping where
almost every edge stays within a group scores high; a grouping that cuts through dense thickets scores
low. Leiden shuffles pixels between communities, greedily, until modularity stops improving, then
refines.

Two knobs control the outcome:

- **`n_neighbors`** (the k from section 2): how local the graph is. Small k builds a fine, fragmented
  graph; large k a smoother, broader one.
- **`resolution`**: how eagerly Leiden splits. It multiplies the penalty for leaving edges *between*
  communities, so higher resolution prefers many small tight groups, lower resolution a few big loose
  ones. This is the dial that sets the granularity of your lipizones.

We learn the clusters on the **control section only**. Control is our reference brain, and in section 5
we will transfer its labels onto the pregnant brain rather than re-cluster it. scanpy wraps the kNN
graph and Leiden in two calls. The Atlas pipeline also starts from a coarse Leiden to find broad lipidomic
domains, then refines them with a divisive splitter into the final lipizones, which you run for real in
section 6. Here we use flat Leiden as the transparent skeleton of the idea.

🔬 **TASK.** Cluster the control pixels into lipizones on their Harmonized embedding. Use the helper
`cl.embedding.leiden_clusters` (it builds a k-nearest-neighbour graph and runs Leiden under the modularity
objective, the recipe the Lipid Brain Atlas uses). Print how many lipizones came out and how big they are.

In [ ]:
# cluster the CONTROL pixels into lipizones on their Harmonized embedding,
# with the Lipid Brain Atlas Leiden recipe (fast, modularity-based, all pixels)
# 🔬 your code here


**check:** Leiden returns a couple dozen lipizones, from a few hundred to several thousand pixels each.
Found with zero supervision: no anatomy, no condition, just the lipid embedding graph.

💡 **HINT.** Open `src/cajal_lipidomics/embedding.py` and read `leiden_clusters`. It is a fast recipe:
a 40-nearest-neighbour graph on the embedding, then leidenalg with the `ModularityVertexPartition`. 

In [ ]:
# not a black box: here is exactly what leiden_clusters runs, unrolled (on a 12k subset for speed)
import leidenalg, igraph as ig
from sklearn.neighbors import NearestNeighbors
sub = np.sort(np.random.default_rng(0).choice(is_control.sum(), size=12000, replace=False))
Xc = G[is_control][sub]
knn = NearestNeighbors(n_neighbors=40, n_jobs=-1).fit(Xc).kneighbors_graph(Xc)
src, dst = knn.nonzero()
g = ig.Graph(n=Xc.shape[0], edges=list(zip(src.tolist(), dst.tolist()))); g.simplify()
unrolled = np.array(leidenalg.find_partition(g, leidenalg.ModularityVertexPartition,
                                             n_iterations=5, seed=230598).membership)
helper_sub = emb.leiden_clusters(Xc)
print("unrolled recipe matches the helper on the subset:", bool((unrolled == helper_sub).all()))

Modularity picks the granularity for you. If you want to turn the dial by hand, pass a `resolution`
to switch to the resolution-tunable partition. We sweep it once just to feel that higher resolution carves
finer lipizones. Nothing about the data changes, only how finely we ask it to be cut.

In [ ]:
# sweep resolution to see it controls granularity (this is the main lipizone dial)
for res in [0.5, 2.0]:
    lab = emb.leiden_clusters(G[is_control], n_neighbors=15, resolution=res, seed=RNG_SEED)
    print(f"resolution {res:>4}  ->  {len(np.unique(lab)):>3} clusters")

**check:** cluster count climbs monotonically with resolution, about ten at 0.3, a couple dozen at
1.0, many more at 3.0. The real Atlas does not reach its hundreds of fine lipizones by turning this dial up, though: it starts
from a coarse Leiden and then refines it with a divisive splitter (section 6), which the paper found more
robust to batch effects than a single high-resolution Leiden. There is no single correct number; the right granularity is a scientific judgement about how
fine a territory is still biologically meaningful.

Before mapping anything, one fast sanity check on the clustering itself: how big are the lipizones? A healthy run gives a spread of sizes, a few large territories and a tail of small ones, never a single blob swallowing the section. `cl.plotting.cluster_size_hist` draws that distribution.

💡 **HINT.** open `src/cajal_lipidomics/plotting.py` and read `cluster_size_hist` before running it. It is three lines (a `value_counts`, then a histogram). Reading the helper you call is the habit this course is built on: you should always know what the code you run actually does.

In [ ]:
# sanity check: how many pixels does each lipizone hold?
pl.cluster_size_hist(leiden_ctrl)
plt.show()

**check:** a right-skewed histogram. Most lipizones hold a few hundred to a couple thousand pixels, with a short tail of larger ones, and none dominates. That balanced spread is what you want before you read any biology into the territories.

## 4. the lipid territories in space, and how they track anatomy

Numbers on a graph mean nothing until we put them back on the brain. We attach the Leiden labels to the
control pixels and colour them on the CCF coordinates. If the clustering captured real biology, the
clusters should be **spatially coherent**: contiguous patches, not salt-and-pepper static, because
neighbouring tissue tends to share lipid composition.

For the colours we use `cl.plotting.lipizone_colors`, which orders the clusters by the similarity of
their lipid centroids and then hands adjacent colours to molecularly similar lipizones. The map reads as
a smooth anatomy rather than random confetti, and the same colour means the same lipizone everywhere we
plot it today.

In [ ]:
# attach the Leiden lipizones to the control AnnData and build a similarity-ordered palette
adata.obs["lipizone"] = "NA"
adata.obs.loc[is_control, "lipizone"] = pd.Series(leiden_ctrl, index=adata.obs_names[is_control]).astype(str)
ctrl_ad = adata[is_control].copy()
color_map = pl.lipizone_colors(ctrl_ad, key="lipizone", rep="X_nmf")   # {label: hex}, similarity-ordered
print("palette built for", len(color_map), "lipizones")

# stash a per-pixel hex colour so the Atlas plotting helpers can read it straight from obs
adata.obs["lipizone_color"] = "#dddddd"
adata.obs.loc[is_control, "lipizone_color"] = (
    pd.Series(leiden_ctrl, index=adata.obs_names[is_control]).astype(str).map(color_map))

zc, yc = z[is_control], y[is_control]
ctrl_colors = pd.Series(leiden_ctrl).astype(str).map(color_map).to_numpy()

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(zc, yc, c=ctrl_colors, s=4, rasterized=True)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
ax.set_title(f"Leiden lipizones on the control section ({n_clusters} clusters)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

**check:** the clusters draw clean, contiguous territories, a ribbon hugging the cortical
surface, a wedge in the white-matter tracts, blocks filling the deep nuclei. They were found from lipid
composition alone, yet they carve the section into anatomy-like shapes. That spatial coherence is the
first sign the lipizones are real and not noise.

Now lay the lipid territories beside the **Allen reference**, the classical anatomy. Each pixel
already carries `allencolor`, the official colour of the Allen region it falls in. Plotting our Leiden
clusters next to the Allen colouring is the eyeball test of how much the two partitions agree.

In [ ]:
# side by side: our Leiden lipizones vs the Allen anatomical regions, same section
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(zc, yc, c=ctrl_colors, s=4, rasterized=True)
axes[0].set_title("Leiden lipizones (from lipids)", fontsize=FS["m"])
axes[1].scatter(zc, yc, c=adata.obs["allencolor"].to_numpy()[is_control], s=4, rasterized=True)
axes[1].set_title("Allen regions (classical anatomy)", fontsize=FS["m"])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

The shapes rhyme but do not match. Some Allen regions split into several lipizones, several Allen
regions merge into one, and a few boundaries land elsewhere entirely.

We can go past the eyeball and ask, region by region, *which* lipizone concentrates in *which* anatomy.
The wrong tool is a plain crosstab of co-occurrence counts: it is dominated by the big categories, since a
huge cortical region racks up large counts with everything simply because it holds many pixels. The right
tool is **reciprocal enrichment**, the measure the Lipid Brain Atlas uses, and we build it here BY HAND so
you see exactly what it does. For a (region, lipizone) pair it asks two questions and multiplies them: how
enriched is this lipizone *inside* this region (its fraction here divided by its average fraction), and how
enriched is this region *inside* this lipizone. A pair scores high only if each is over-represented in the
other, which cancels the size bias.

In [ ]:
# build it BY HAND on a toy first, so every step is transparent.
# clusters A/B and regions X/Y, with A mostly in X and B mostly in Y
clu = pd.Series(list("AAAABBBBAB"), name="cluster")
reg = pd.Series(list("XXXXYYYYYX"), name="region")

# direction 1: how enriched is each cluster INSIDE each region?
c1 = pd.crosstab(reg, clu)                 # counts, region x cluster
n1 = c1 / c1.sum()                         # each cluster column -> fraction across regions
n1 = (n1.T / n1.T.mean()).T                # divide each region row by its mean -> enrichment
# direction 2: how enriched is each region INSIDE each cluster?
c2 = pd.crosstab(clu, reg)
n2 = c2 / c2.sum()
n2 = (n2.T / n2.T.mean()).T
# reciprocal enrichment = the two directions multiplied
toy_recip = n1 * n2.T                       # region x cluster
print("counts:\n", c1.values)
print("\nreciprocal enrichment (region x cluster):\n", toy_recip.round(2))

**check:** A scores high in X and B in Y, near 1 (no enrichment) off-diagonal, because each cluster sits
in its own region and the product rewards only mutual over-representation. The exact same three steps,
two crosstabs turned into enrichments and multiplied, are packaged as `cl.multimodal.reciprocal_enrichment`;
open `src/cajal_lipidomics/multimodal.py` and read it to confirm it is just what you wrote here.

In [ ]:
# now on the real control section, unrolled exactly as in the Lipid Brain Atlas (001-IDCARDS).
from scipy.cluster import hierarchy as sch
allen_ctrl = adata.obs["acronym"].astype(str).to_numpy()[is_control]   # Allen region per control pixel
reg = pd.Series(allen_ctrl, name="region"); lip = pd.Series(leiden_ctrl, name="lipizone")
common = reg.value_counts().index[reg.value_counts() > 500]   # regions with enough pixels to mean anything
keep = reg.isin(common).to_numpy(); reg, lip = reg[keep], lip[keep]

c1 = pd.crosstab(reg, lip); n1 = c1 / c1.sum(); n1 = (n1.T / n1.T.mean()).T   # lipizone enriched in region
c2 = pd.crosstab(lip, reg); n2 = c2 / c2.sum(); n2 = (n2.T / n2.T.mean()).T   # region enriched in lipizone
enr = n1 * (n2.T)                                                              # reciprocal, region x lipizone

# order columns by hierarchical clustering (weighted linkage, optimal leaf order), rows by their argmax
col_order = sch.leaves_list(sch.linkage(sch.distance.pdist(enr.T), method="weighted", optimal_ordering=True))
enr = enr.iloc[:, col_order]
enr = enr.iloc[np.argsort(np.argmax(enr.values, axis=1)), :]

# Luca's display: raw enrichment, vmin/vmax at the 2nd/98th percentiles (NO log scale), Purples
import seaborn as sns
plt.figure(figsize=(10, 7))
sns.heatmap(enr, cmap="Purples", cbar_kws={"label": "enrichment"}, xticklabels=False, yticklabels=True,
            vmin=np.percentile(enr.values, 2), vmax=np.percentile(enr.values, 98))
plt.xlabel("lipizone"); plt.ylabel("Allen region"); plt.title("reciprocal enrichment: region x lipizone")
plt.yticks(rotation=0); plt.tight_layout(); plt.show()

**check:** a Purples block matrix, dark where a lipizone is enriched in a region above chance. The leaf
ordering pulls the dark cells toward a diagonal: each lipizone concentrates in one or two Allen regions, its
home territory, and stays pale elsewhere. Reading a dark row, a single region often carries several enriched
lipizones, the lipidome resolving one anatomical region into finer compositional sub-territories. The 2nd/98th
percentile limits keep a couple of extreme cells from washing out the rest, and because the score is a ratio
corrected in both directions, it is honest about the very different sizes of regions and lipizones.

To name a lipid territory you ask which lipids define it. The helper `cl.analysis.marker_lipids`
runs a one-versus-rest comparison per cluster and ranks the lipids most elevated inside it. We attach
the Leiden labels to the uMAIA lipid matrix and read the markers of the largest clusters. (Open `src/cajal_lipidomics/analysis.py` and read `marker_lipids` to see the one-versus-rest fold change it computes.)

In [ ]:
# marker lipids per Leiden cluster (one-vs-rest log2 fold change), control only
adata_ctrl = adata[is_control].copy()
adata_ctrl.X = adata_ctrl.layers["umaia"]                 # markers come from the real lipid intensities
adata_ctrl.var_names = adata_ctrl.var["lipid"].values
adata_ctrl.var_names_make_unique()
adata_ctrl.obs["leiden"] = pd.Categorical(leiden_ctrl)
markers = an.marker_lipids(adata_ctrl, "leiden", top_n=4)

# show markers for the four largest clusters
biggest = pd.Series(leiden_ctrl).value_counts().head(4).index
for cluster_id in biggest:
    top = markers.get(cluster_id, [])
    pretty = ", ".join(f"{name} (log2FC {lfc:+.2f})" for name, lfc in top)
    print(f"cluster {cluster_id:>2} ({(leiden_ctrl == cluster_id).sum():>5} px):  {pretty}")

# also surface the WHITE-MATTER territories so the HexCer claim is visible in this cell's own output:
# the clusters whose top marker is a HexCer (a myelin sphingolipid)
wm_clusters = [cid for cid in markers if markers[cid] and markers[cid][0][0].startswith("HexCer")]
print("\nwhite-matter (HexCer-headed) lipizones:")
for cluster_id in wm_clusters:
    top = markers.get(cluster_id, [])
    pretty = ", ".join(f"{name} (log2FC {lfc:+.2f})" for name, lfc in top)
    print(f"cluster {cluster_id:>2} ({(leiden_ctrl == cluster_id).sum():>5} px):  {pretty}")

**check:** each cluster prints its top lipids with a positive log2 fold change versus the rest of
the brain. The white-matter territories are headed by **HexCer** (a hexosylceramide, a myelin
sphingolipid); other territories are led by phosphatidylcholines such as PC 32:0 and by a few
unannotated ions that the annotation notebook could not match to a reference lipid. The markers are how
a nameless "cluster 14" becomes a biologically meaningful lipizone: it is defined by the lipids it
concentrates.

The printed markers name each territory one cluster at a time. To see the whole picture at once, the Atlas builds a **lipizone x lipid heatmap**: every lipizone is a row, every lipid a column, each cell the mean intensity of that lipid in that lipizone. `cl.plotting.sorted_lipid_heatmap` then orders both rows and columns by cosine similarity with optimal leaf ordering, so chemically similar lipizones sit together and co-varying lipids form blocks. The sorting is what turns a wall of numbers into readable structure.

💡 **HINT.** open `src/cajal_lipidomics/plotting.py` and read `sorted_lipid_heatmap`: it 0-1 normalizes each lipid, averages by your group key, then optimal-leaf-orders both axes on cosine distance. That ordering convention is reused for every heatmap in the course, so it is worth reading once.

In [ ]:
# lipizone x lipid: mean intensity per lipizone, both axes cosine optimal-leaf-ordered
pl.sorted_lipid_heatmap(adata_ctrl, group_key="leiden", figsize=(16, 7),
                        title="lipizone x lipid (sorted)")
plt.show()

**check:** a block-structured heatmap, not random speckle. Bands of lipids light up together across groups of neighbouring lipizones, the molecular signatures the marker table listed cluster by cluster. Reading down a column you see one lipid's territory; reading across a row you read a lipizone's recipe. This is the canonical view of what a lipizone *is*: a characteristic lipid composition.

## 5. label transfer: carry the control lipizones onto the pregnant brain

We clustered the control section. We do **not** want to cluster the pregnant section separately and then
puzzle out which of its new clusters matches which control cluster. We want to *carry the existing
labels over* so both sections speak one vocabulary and we can compare like with like.

The idea is the same kNN trick as Leiden, used differently. For each **query** pixel (a pregnant pixel,
unlabelled) find its nearest **reference** pixels (control pixels, already labelled) in the shared
embedding, and take a **majority vote** of their labels. The pixel inherits whatever its neighbours
mostly are. A bonus falls out for free: how *unanimous* the neighbours are gives a **confidence**. A
pregnant pixel whose 15 control neighbours all carry one lipizone is a confident call; one split
eight-to-seven is a flag to inspect.

The catch is that a kNN vote only makes sense if control and pregnant pixels live in one comparable
coordinate system. The pregnant section was acquired on a different day, so its absolute factor values
drift even where the biology is identical. If we voted in that drifted space, every pregnant pixel would
just find its nearest *pregnant* neighbours and the transfer would be meaningless. This is exactly the
section offset **Harmony** removed when the embedding notebook produced `X_harmony`, harmonizing on
section only (not on condition, so the pregnancy biology stays intact). We vote in that already-shared
space.

In [ ]:
# split the shared (Harmonized) space into reference (control) and query (pregnant)
Z_ref,  y_ref = G[is_control], leiden_ctrl                 # control + its Leiden lipizones
Z_query       = G[is_pregnant]                             # pregnant, no labels yet
print(f"reference: {Z_ref.shape[0]} labelled control pixels")
print(f"query    : {Z_query.shape[0]} pregnant pixels to annotate")

🔬 **TASK.** Do the transfer by hand with scikit-learn's `KNeighborsClassifier` so you see there
is no magic. Fit it on the labelled control reference, predict on the pregnant query, and read off the
neighbour-vote confidence as the max class probability per pixel.

In [ ]:
# kNN majority vote: fit on control, transfer onto pregnant
# 🔬 your code here


**check:** every pregnant pixel gets a lipizone, with a mean neighbour-vote fraction around 86%. Most pixels sit deep inside a
territory and vote nearly unanimously, while a minority near boundaries split their vote and come out
lower — those are the ones a curator would review.

💡 **HINT.** `cl.embedding.knn_transfer(Z_ref, y_ref, Z_query, k=15)` packs exactly this fit-predict-
confidence into one call and returns `(pred, conf)`. We unrolled it so the majority vote is visible. Open `src/cajal_lipidomics/embedding.py` and read `knn_transfer` to confirm the helper is exactly this fit-predict-confidence.

In [ ]:
# the helper does the same vote in one line
pred_h, conf_h = emb.knn_transfer(Z_ref, y_ref, Z_query, k=15)
print("helper agrees with hand-rolled prediction on",
      f"{(pred_h == pred_pregnant).mean()*100:.1f}% of pregnant pixels")

Now the payoff: control and pregnant, side by side, in the **same lipizone vocabulary**. The
control panel shows the Leiden lipizones we learned; the pregnant panel shows those same labels
transferred by the kNN vote, drawn with the same similarity-ordered palette. Because the colours mean
the same thing in both, any shift in a territory's size or position is a candidate biological change.

In [ ]:
# store the transferred labels, then paint both sections with one shared palette
adata.obs.loc[is_pregnant, "lipizone"] = pd.Series(pred_pregnant, index=adata.obs_names[is_pregnant])
preg_colors = pd.Series(pred_pregnant).astype(str).map(color_map).to_numpy()
adata.obs.loc[is_pregnant, "lipizone_color"] = (
    pd.Series(pred_pregnant, index=adata.obs_names[is_pregnant]).astype(str).map(color_map))
zp, yp = z[is_pregnant], y[is_pregnant]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(zc, yc, c=ctrl_colors, s=4, rasterized=True)
axes[0].set_title("control: lipizones learned with Leiden", fontsize=FS["m"])
axes[1].scatter(zp, yp, c=preg_colors, s=4, rasterized=True)
axes[1].set_title("pregnant: same lipizones, transferred by kNN vote", fontsize=FS["m"])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

Lipizones are not a flat list, they come from a tree of splits. Rather than jump straight to the
terminal leaves, watch the section divide: the first split paints **2** colours (each region the modal
colour of its leaves), the next **4**, then **8**, then **16**. This is how the Lipid Brain Atlas reads a
hierarchy, and it shows which coarse division each fine lipizone descends from.

In [ ]:
from cajal_lipidomics import plotting as _pl
ctrl_ad = adata[is_control].copy()
levels = _pl.lipizone_hierarchy_levels(ctrl_ad, key='lipizone', rep='X_nmf', max_level=4)
_pl.plot_hierarchy_levels(ctrl_ad, levels, color_key='lipizone_color', section_key='SectionID')
plt.suptitle('the lipizone hierarchy, split by split: 2 -> 4 -> 8 -> 16 colours', y=1.02); plt.show()

To reason about a single object in space, isolate it: one lipizone in **red** over a grey cast of all
tissue. Where exactly does this one territory sit, and is it bilateral and contiguous as a real anatomical
object should be?

In [ ]:
from cajal_lipidomics import plotting as _pl
biggest = adata.obs.loc[is_control, 'lipizone'].value_counts().index[0]
_pl.highlight_lipizone(adata, biggest, key='lipizone', section_key='SectionID')
plt.show()

Finally, zoom in. A **mosaic** of one region, points enlarged, shows how lipizones tile a place of
interest, the texture you cannot see in the whole-section view. Change the box to explore your own region.

In [ ]:
from cajal_lipidomics import plotting as _pl
xs = adata.obs.loc[is_control, 'x']; ys = adata.obs.loc[is_control, 'y']
# a box around the dorsal third (cortex + hippocampus); edit to roam
box_x = (xs.quantile(0.25), xs.quantile(0.75)); box_y = (ys.quantile(0.05), ys.quantile(0.45))
sid = sorted(adata.obs.loc[is_control,'SectionID'].unique())[0]
_pl.mosaic(adata, lipizone_color_key='lipizone_color', section=sid, xlim=box_x, ylim=box_y, point_size=22)
plt.show()

**check:** the two sections show the same coloured territories in the same anatomical places, the
cortical ribbon, the white-matter wedge, the deep nuclei all reappearing in matching colours. That
correspondence is the whole point of label transfer: we annotated the pregnant brain without clustering
it, just by asking each pixel which control pixels it resembles. The two are now directly comparable.

We have the lipizones, and from the previous notebook we have a t-SNE of the lipid embedding (`adata.obsm["X_tsne"]`, a 2-D layout that places pixels with similar lipid programs near each other). Now that the lipizones are *earned* from our own clustering and transfer, not borrowed from the paper, we can colour that t-SNE two ways on the same coordinates: first by our lipizone colours, then by the Allen region colour.

💡 **HINT.** open `src/cajal_lipidomics/plotting.py` and read `tsne_colored`. With `kind="color"` it scatters `obsm["X_tsne"]` and reads a per-pixel hex colour straight from the obs column you name. We pass `lipizone_color` (ours, built and transferred above) and `allencolor` (anatomy).

In [ ]:
# colour the embedding t-SNE by your lipizones, then by Allen anatomy, on the same layout
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
pl.tsne_colored(adata, by="lipizone_color", kind="color", ax=axes[0],
                title="t-SNE by your lipizones")
pl.tsne_colored(adata, by="allencolor", kind="color", ax=axes[1],
                title="t-SNE by Allen region")
plt.tight_layout(); plt.show()

**check:** the lipizone-coloured t-SNE breaks into clean, well-separated islands, as it should, since the t-SNE and the lipizones both read the same lipid embedding. The Allen-coloured version of the very same points is patchier: anatomical regions spread across several lipizone islands, and some islands mix Allen colours. It is the ARI lesson again, now seen in the embedding rather than in space, lipid identity and classical anatomy overlapping without being the same partition.

## 6. the production engine: run EUCLID for real

What you built is the honest skeleton of how the Lipid Brain Atlas finds and transfers lipizones: a kNN
graph, a community cut, a kNN vote across a Harmonized space. The published Atlas uses a more
sophisticated engine called **EUCLID** (Enhanced uMAIA for Clustering Lipizones, Imputation, and
Differential analysis). Now you run it yourself, on your control brain, and put its lipizones next to
the ones you made by hand.

Where we ran **one flat Leiden** at a single resolution, EUCLID runs a **divisive, top-down binary
splitter**. It starts with all pixels in one group and recursively asks "should this group split in
two?". At each candidate split it recomputes a *local* NMF on just the pixels in that branch, so fine
structure the global 12 factors blur out becomes visible; it over-segments into K micro-clusters with
k-means, then re-aggregates those into exactly two groups. A split is accepted only if it passes three
gates: the two halves differ in at least a couple of lipids (a Mann-Whitney test), each half is big
enough, and each half is spatially coherent. The result is a tree, and the terminal leaves are the
lipizones, organized into a class, subclass, supertype hierarchy.

Where we transferred labels with a kNN vote, EUCLID trains an **XGBoost classifier at every node** of
that tree. To annotate a new pixel it walks the tree from the root, at each node applying the local NMF
and the trained classifier to decide left or right, until the pixel lands in a leaf. That is how the
Atlas label-transferred the lipizone hierarchy onto the pregnant brains.

We run a small tree (shallow `max_depth`) so the whole divisive recursion finishes in a couple of
minutes instead of hours.

In [ ]:
# EUCLID lives on GitHub; clone it and put it on the import path.
import os, sys, subprocess, types

# clone the package (a shallow clone is enough) and add its src/ to sys.path
EUCLID_DIR = os.path.abspath("../../external/EUCLID")
if not os.path.isdir(EUCLID_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/lamanno-epfl/EUCLID.git", EUCLID_DIR], check=True)
sys.path.insert(0, os.path.join(EUCLID_DIR, "src"))

# EUCLID's clustering module imports a few small, pure-Python packages that are not in the course
# env (EUCLID is an external package). Install them once if missing -- they are lightweight and do
# NOT disturb the pinned numpy/pandas.
for _pkg, _mod in [("imbalanced-learn", "imblearn"), ("kneed", "kneed"), ("PyPDF2", "PyPDF2")]:
    try:
        __import__(_mod)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

# clustering.py also does `import squidpy as sq` at the top. squidpy is a heavy geospatial
# dependency (it pulls in pyproj/PROJ and forces numpy>=2): it fails to build on a clean Mac and
# would break this pinned env. This notebook never uses squidpy -- it only runs EUCLID's divisive
# clustering -- so instead of installing it we register a tiny stub that satisfies the import and
# raises a clear error only if a squidpy function is ever actually called.
if "squidpy" not in sys.modules:
    _sq = types.ModuleType("squidpy")
    def _no_squidpy(*a, **k):
        raise RuntimeError("squidpy is intentionally not installed for this course; this code "
                           "path (squidpy spatial autocorrelation) is not used in the notebook.")
    class _Stub:
        def __getattr__(self, _):
            return _no_squidpy
    _sq.gr = _Stub()
    sys.modules["squidpy"] = _sq

# EUCLID's package __init__ eagerly imports every submodule (embedding -> mofapy2,
# preprocessing -> squidpy, ...) and downloads reference data on import. We only need the divisive
# clustering, so we replace __init__ with a minimal stub: `from euclid_msi import clustering` then
# loads clustering.py (and its backSPIN helper) in isolation, skipping the unrelated heavy
# dependencies and the download. We also strip clustering.py's stray `print("ciao")` debug line so
# the apply step's output stays clean. Both patches are deterministic, so re-running is safe.
open(os.path.join(EUCLID_DIR, "src", "euclid_msi", "__init__.py"), "w").write('__version__ = "0.0.1b"\n')
_csrc = os.path.join(EUCLID_DIR, "src", "euclid_msi", "clustering.py")
_txt = open(_csrc).read()
open(_csrc, "w").write(_txt.replace('        print("ciao")\n', ""))

from euclid_msi import clustering as euc
print("EUCLID imported from", EUCLID_DIR)

# one small compatibility shim: newer xgboost moved `early_stopping_rounds` from .fit() to the
# constructor, so we bridge EUCLID's older call. The clustering logic is untouched.
import xgboost as _xgb
class _XGB(_xgb.XGBClassifier):
    def fit(self, X, y, eval_set=None, early_stopping_rounds=None, verbose=False, **kw):
        if early_stopping_rounds is not None and eval_set is not None:
            self.set_params(early_stopping_rounds=early_stopping_rounds)
        return super().fit(X, y, eval_set=eval_set, verbose=verbose, **kw)
euc.XGBClassifier = _XGB

EUCLID's `Clustering` object expects an embedding bundle: the lipid matrix it runs local NMF on,
the standardized global embedding it carries as context at every node, the per-pixel spatial
coordinates, and a section column. We assemble that for the control brain.

One practical point. The splitter's spatial-coherence gate was built for a *stack* of serial coronal
sections along the rostrocaudal axis; we have a single coronal plane, so that rostrocaudal continuity
is impossible here. Rather than fake a section stack, we mark all pixels as one section (EUCLID then
falls back to a plain random train/val/test split) and bypass the continuity gate, and we run on
every control pixel.

In [ ]:
# assemble EUCLID's input bundle for the control brain -- ALL control pixels, no subsampling
from sklearn.preprocessing import StandardScaler

ctrl_full = adata[is_control]
rng = np.random.default_rng(RNG_SEED)

lipid_names = ctrl_full.var["lipid"].values
X_lip = ctrl_full.layers["umaia"].astype(np.float64)             # raw uMAIA lipids
X_approx = X_lip + np.abs(rng.normal(0, 1e-6, X_lip.shape))      # tiny POSITIVE jitter: no flat lipid,
                                                                # and stays >= 0 (EUCLID's local NMF needs non-negative input)

# EUCLID's continuity gate and its train/val/test split were built for a STACK of coronal sections
# along the rostrocaudal axis. We have a SINGLE coronal section, so that rostrocaudal continuity is
# impossible -- the original atlas simply does not apply it here. So we (a) mark all pixels as one
# section, which makes EUCLID fall back to a plain random 60/20/20 train/val/test split, and
# (b) bypass the rostrocaudal continuity check so it never blocks a split. No fake dorsoventral
# "sections" -- that would be drift from the original.
euc.Clustering._continuity_check = lambda self, spat, kmeans_labels, **kw: (True, True, 1, 1, 1e9, 1e9)

# a minimal embedding object carrying exactly what Clustering reads
emb_bundle = type("Emb", (), {})()
ae = ad.AnnData(X=X_lip.copy())
ae.var_names = lipid_names; ae.var_names_make_unique()
ae.obs_names = ctrl_full.obs_names
ae.obs["x"] = ctrl_full.obs["zccf"].to_numpy()
ae.obs["y"] = ctrl_full.obs["yccf"].to_numpy()
ae.obs["SectionID"] = 0                                          # one real coronal section
ae.obsm["X_Harmonized"] = ctrl_full.obsm["X_harmony"].astype(np.float64)   # global context per node
ae.obsm["X_approximated"] = X_approx                                       # local NMF runs on this
ae.obsm["X_TSNE"] = ctrl_full.obsm["X_tsne"].astype(np.float64)
emb_bundle.adata = ae

clust = euc.Clustering(emb_bundle, analysis_name="euclid_course")

# the differential-lipid gate compares groups with Mann-Whitney; clean the percentile-clipped
# lipid table so it never receives NaN (a few flat lipids clip to NaN) and add tiny jitter
L = clust.adatamaia.obsm["lipids"]
clust.adatamaia.obsm["lipids"] = pd.DataFrame(
    np.nan_to_num(L.values, nan=0.0) + rng.normal(0, 1e-6, L.shape),
    index=L.index, columns=L.columns)
print("EUCLID input ready:", ctrl_full.n_obs, "control pixels x", len(lipid_names), "lipids")

🔬 **TASK.** Run `learn_euclid_clustering` on the control brain. This is the real divisive tree
learner: at each node a local NMF, k-means over-segmentation, backSPIN re-aggregation to two groups (backSPIN is a biclustering routine
that sorts the micro-clusters along a gradient and finds the cleanest place to cut them in two), the
three acceptance gates, and an XGBoost classifier. We pass a deep `max_depth` and run on every control pixel; it
finishes in about five minutes and resolves ~126 lipizones.

In [ ]:
# LEARN the divisive lipizone tree on ALL control pixels (this actually runs; DEEP tree,
# ~5 minutes, ~126 terminal lipizones)
root_node, clustering_log = clust.learn_euclid_clustering(
    K=15,                # over-segment each branch into 30 micro-clusters before re-aggregating to 2
    min_voxels=100,      # do not split a group smaller than 100 pixels
    min_diff_lipids=2,   # accept a split only if >2 lipids differ (Mann-Whitney + BH)
    min_fc=0.2, pthr=0.05,
    ACCTHR=0.6,          # the per-node XGBoost must reach 0.6 test accuracy to keep the split
    max_depth=7,         # deep tree: up to 2^7=128 leaves (gates prune some branches)
    combinations=2,      # how many top-factor combinations to try per split
    xgb_n_estimators=40, xgb_n_jobs=8,
    do_plotting=False,
)

# roll the per-level 1/2 decisions into a lipizone barcode per pixel (EUCLID's convention)
lvl_cols = [c for c in clustering_log.columns if c.startswith("level_")]
euclid_leaf = clustering_log[lvl_cols].fillna(0).astype(int).astype(str).agg("".join, axis=1)
print("EUCLID learned", euclid_leaf.nunique(), "lipizones on the control brain")
print(euclid_leaf.value_counts())

**check:** EUCLID prints a handful of lipizones, each a
barcode of the 1/2 decisions taken down the tree, like `1212`. It is fewer than your two dozen-odd Leiden clusters
because we capped the depth; let the tree grow deeper and the leaf count multiplies. The point is that
this is the genuine package running its genuine divisive algorithm on your data.

Now **apply** the learned tree to the full data, both sections, by walking each pixel from the
root down to a leaf. This is EUCLID's label transfer: the XGBoost classifier at each node decides left
or right, the exact divisive cousin of your kNN vote in section 5.

In [ ]:
# APPLY the learned tree onto the FULL data (control + pregnant) by root-to-leaf descent
full_ad = ad.AnnData(X=adata.layers["umaia"].astype(np.float64))
full_ad.var_names = adata.var["lipid"].values; full_ad.var_names_make_unique()
full_ad.obs_names = adata.obs_names

# standardized global embedding for the full data (same standardization EUCLID used internally)
global_full = pd.DataFrame(StandardScaler().fit_transform(adata.obsm["X_harmony"]),
                           index=adata.obs_names)

paths_df = clust.apply_euclid_clustering(root_node, full_ad, global_full)
ep_cols = [c for c in paths_df.columns if c.startswith("level_")]
euclid_lipizone = paths_df[ep_cols].fillna(0).astype(int).astype(str).agg("".join, axis=1)
adata.obs["euclid_lipizone"] = euclid_lipizone.reindex(adata.obs_names).values
print("EUCLID lipizones on the full data:", adata.obs["euclid_lipizone"].nunique())
print(adata.obs["euclid_lipizone"].value_counts())

**check:** every pixel of both sections now carries an EUCLID lipizone, the same small set of
leaves the tree learned, transferred by the per-node classifiers. The counts are balanced across the two
sections, which is the divisive analogue of the matching territories your kNN transfer produced.

Put the two clusterings side by side on the control section: your hand-built Leiden lipizones on
the left, EUCLID's divisive lipizones on the right, each with its own similarity-ordered palette. They
are not meant to match label-for-label (EUCLID is coarser here because we capped the depth) but the
large-scale territories, cortex, white matter, deep nuclei, should reappear in both.

In [ ]:
# Leiden (yours) vs EUCLID (the package), same control section
euclid_ctrl = adata.obs["euclid_lipizone"].to_numpy()[is_control]
euclid_cmap = pl.lipizone_colors(adata[is_control].copy(), key="euclid_lipizone", rep="X_nmf")
euclid_colors = pd.Series(euclid_ctrl).astype(str).map(euclid_cmap).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
axes[0].scatter(zc, yc, c=ctrl_colors, s=4, rasterized=True)
axes[0].set_title(f"your Leiden lipizones ({n_clusters})", fontsize=FS["m"])
axes[1].scatter(zc, yc, c=euclid_colors, s=4, rasterized=True)
axes[1].set_title(f"EUCLID lipizones ({adata.obs['euclid_lipizone'].nunique()}, deep tree)",
                  fontsize=FS["m"])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

**check:** both panels carve the section into contiguous territories, and the big divisions line
up, the white-matter wedge and the cortical ribbon are visible in each. EUCLID resolves many more, finer
territories here (about 126, against your ~18 Leiden lipizones), because we ran the full deep tree on every pixel; the algorithm is the careful, transferable cousin of
the flat Leiden you ran in section 3, and you just ran it for real.

## 7. save your lipizones for the next notebook

The lipizones we carry forward are the transparent ones you can fully account for: the control Leiden
clusters and their kNN transfer onto the pregnant brain, already stored in `adata.obs["lipizone"]`. We
keep the EUCLID labels alongside as `euclid_lipizone` for comparison. The next notebook loads this file
and asks how the lipidome remodels in pregnancy, lipizone by lipizone.

In [ ]:
# the lipizone column already holds control-Leiden + pregnant-transfer; make it categorical and save
adata.obs["lipizone"] = pd.Categorical(adata.obs["lipizone"].astype(str))
# h5ad needs consistent dtypes: cast any object columns (colours, labels) to plain strings
for col in adata.obs.columns:
    if adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].astype(str)
print("lipizones across both sections:", adata.obs["lipizone"].nunique())
print(adata.obs.groupby("Condition", observed=True)["lipizone"].nunique())

out = "../../data/derived/06_clustered.h5ad"
adata.write_h5ad(out)
print("saved", out, "->", adata.shape)

**check:** `06_clustered.h5ad` is written, carrying `obs["lipizone"]` (your control lipizones,
transferred to pregnant) and `obs["euclid_lipizone"]` (the package's version) for every pixel of both
sections. That is the staged input the differential notebook will pick up.

## what you did

You turned a 12-dimensional lipid embedding into named brain territories and carried those names across
conditions.

- You built a **kNN graph** by hand and saw it is just a table of who-is-near-whom in lipid space, then
  let **Leiden** cut it into communities, your **lipizones**, with **resolution** setting how fine they
  are.
- You painted the lipizones on the brain (spatially coherent territories), laid them beside the Allen
  reference, and measured agreement with the **Adjusted Rand Index**. The ARI near 0.06 is the lesson:
  lipizones partially track anatomy but are their own organizing principle, because membrane lipid
  composition reflects biology that classical anatomy does not capture.
- You ran **kNN label transfer** across the **Harmony**-shared space (harmonized on section, so the
  pregnancy biology survives), copying the control lipizones onto the pregnant brain by majority vote,
  with a free **confidence** map that lit up the uncertain boundaries.
- You **cloned EUCLID and ran it for real**: its divisive tree with per-node XGBoost classifiers does a
  more careful, transferable version of the same learn-then-transfer logic, and its lipizones sat next
  to yours on the same section.